# Census API Demographic Analysis
### STRAT-412 | Sprouts, Trader Joe's & Whole Foods Growth Strategy

This notebook pulls **zip-code level demographic data** from the U.S. Census Bureau's American Community Survey (ACS) 5-Year Estimates and merges it with the store location CSVs to identify the demographic profiles each grocery chain targets.

**Key variables pulled:**
- `B19013_001E` — Median Household Income
- `B01003_001E` — Total Population
- `B15003_022E` — Bachelor's degree holders (25+)
- `B15003_023E` — Master's degree holders (25+)
- `B15003_025E` — Doctorate degree holders (25+)
- `B15003_001E` — Total population 25+ (denominator for education %)
- `B01002_001E` — Median Age

**API Docs:** https://api.census.gov/data/2022/acs/acs5/variables.html

In [ ]:
# ─────────────────────────────────────────────
# CELL 0: Install & Imports
# ─────────────────────────────────────────────
# The census and pandas libraries are needed.
# requests comes pre-installed in Colab.
!pip install census us pandas --quiet

import requests
import pandas as pd
import time
import re
from google.colab import files

# ── Census API key ──────────────────────────────────────────────────────────
# Get a FREE key in ~30 seconds at: https://api.census.gov/data/key_signup.html
# Paste it below. Without a key you get ~500 requests/day (usually enough).
CENSUS_API_KEY = ""   # <-- paste your key here, or leave empty for keyless access

# ACS 5-year dataset year (latest available as of 2025)
ACS_YEAR = "2022"

# Variables to pull for each zip code
VARIABLES = ",".join([
    "B19013_001E",   # Median household income
    "B01003_001E",   # Total population
    "B01002_001E",   # Median age
    "B15003_001E",   # Population 25+ (education denominator)
    "B15003_022E",   # Bachelor's degree
    "B15003_023E",   # Master's degree
    "B15003_025E",   # Doctorate degree
])

print("Setup complete.")

In [ ]:
# ─────────────────────────────────────────────
# CELL 1: Helper — Fetch Demographics for a Batch of Zip Codes
# ─────────────────────────────────────────────
# The Census API allows up to 50 geo entries per request.
# We batch zip codes in groups of 50 to stay within limits.

CENSUS_BASE = "https://api.census.gov/data/" + ACS_YEAR + "/acs/acs5"

def fetch_zip_demographics(zip_codes):
    """
    Given a list of 5-digit zip code strings, returns a DataFrame with
    demographic variables from ACS 5-year estimates.
    Batches requests in groups of 50.
    """
    all_rows = []
    unique_zips = list(set(str(z).zfill(5) for z in zip_codes if str(z).strip()))

    # Process in batches of 50
    batch_size = 50
    for i in range(0, len(unique_zips), batch_size):
        batch = unique_zips[i:i + batch_size]
        zip_str = ",".join(batch)

        params = {
            "get": "NAME," + VARIABLES,
            "for": "zip code tabulation area:" + zip_str,
        }
        if CENSUS_API_KEY:
            params["key"] = CENSUS_API_KEY

        try:
            resp = requests.get(CENSUS_BASE, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            # First row is column headers
            headers = data[0]
            for row in data[1:]:
                all_rows.append(dict(zip(headers, row)))
        except Exception as e:
            print("Error fetching batch starting at index " + str(i) + ": " + str(e))

        time.sleep(0.5)  # Be polite to the Census API
        print("Fetched batch " + str(i // batch_size + 1) + " of " + str(-(-len(unique_zips) // batch_size)))

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)

    # Rename columns to readable names
    col_map = {
        "zip code tabulation area": "zip",
        "B19013_001E": "median_income",
        "B01003_001E": "total_population",
        "B01002_001E": "median_age",
        "B15003_001E": "pop_25plus",
        "B15003_022E": "bachelors",
        "B15003_023E": "masters",
        "B15003_025E": "doctorate",
    }
    df = df.rename(columns=col_map)

    # Convert numeric columns (Census returns strings; -666666666 = missing)
    numeric_cols = ["median_income", "total_population", "median_age",
                    "pop_25plus", "bachelors", "masters", "doctorate"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].where(df[col] != -666666666, other=None)

    # Calculate college+ attainment rate
    if all(c in df.columns for c in ["bachelors", "masters", "doctorate", "pop_25plus"]):
        df["college_plus"] = df["bachelors"] + df["masters"] + df["doctorate"]
        df["college_plus_pct"] = (df["college_plus"] / df["pop_25plus"] * 100).round(1)

    return df


# Quick test with a few zip codes
test_zips = ["90210", "10001", "60614", "77002", "98101"]
test_df = fetch_zip_demographics(test_zips)
print(test_df[["zip", "median_income", "total_population", "median_age", "college_plus_pct"]].to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# CELL 2: Load Store CSVs
# ─────────────────────────────────────────────
# Upload sprouts_stores.csv, trader_joes_stores.csv, whole_foods_stores.csv
# from your local machine using the Colab file uploader.

print("Please upload your 3 store CSV files when prompted...")
uploaded = files.upload()   # Colab file picker — select all 3 CSVs

import io

# ── Load each CSV ────────────────────────────────────────────────────────────
# Adjust filenames if yours differ

sprouts_df   = pd.read_csv(io.BytesIO(uploaded.get("sprouts_stores.csv",   b"") or list(uploaded.values())[0]))
tj_df        = pd.read_csv(io.BytesIO(uploaded.get("trader_joes_stores.csv", b"") or list(uploaded.values())[1]))
wf_df        = pd.read_csv(io.BytesIO(uploaded.get("whole_foods_stores.csv", b"") or list(uploaded.values())[2]))

# Tag each with chain name
sprouts_df["chain"]  = "Sprouts"
tj_df["chain"]       = "Trader Joes"
wf_df["chain"]       = "Whole Foods"

print("Sprouts:     " + str(len(sprouts_df)) + " stores")
print("Trader Joes: " + str(len(tj_df)) + " stores")
print("Whole Foods: " + str(len(wf_df)) + " stores")
print()
print("Sprouts columns:", list(sprouts_df.columns))
print("TJ columns:",      list(tj_df.columns))
print("WF columns:",      list(wf_df.columns))

In [ ]:
# ─────────────────────────────────────────────
# CELL 3: Normalize Zip Column & Combine
# ─────────────────────────────────────────────
# All three CSVs should have a 'Zip' or 'zip' column.
# Standardize to 5-digit zero-padded strings.

def clean_zip(series):
    """Extract first 5 digits from zip column, zero-pad, return as string."""
    return (
        series.astype(str)
              .str.extract(r"(\d{5})", expand=False)
              .str.zfill(5)
    )

# Rename zip columns to lowercase 'zip' for consistency
for df, name in [(sprouts_df, "Sprouts"), (tj_df, "Trader Joes"), (wf_df, "Whole Foods")]:
    # Try common column name variants
    zip_col = None
    for c in df.columns:
        if c.lower() in ["zip", "zipcode", "zip_code", "postal_code", "zip code"]:
            zip_col = c
            break
    if zip_col:
        df["zip"] = clean_zip(df[zip_col])
        print(name + ": using column '" + zip_col + "' as zip")
    else:
        print("WARNING: No zip column found for " + name + ". Columns: " + str(list(df.columns)))

# Combine into one master dataframe
all_stores = pd.concat([sprouts_df, tj_df, wf_df], ignore_index=True)
print()
print("Total stores combined: " + str(len(all_stores)))
print("Unique zip codes: " + str(all_stores["zip"].nunique()))

In [ ]:
# ─────────────────────────────────────────────
# CELL 4: Fetch Demographics for All Store Zip Codes
# ─────────────────────────────────────────────
# This hits the Census API with all unique zip codes across all 3 chains.
# Expect ~1,500 unique zips; will take ~1-2 minutes with batching.

all_zips = all_stores["zip"].dropna().unique().tolist()
print("Fetching demographics for " + str(len(all_zips)) + " unique zip codes...")

demo_df = fetch_zip_demographics(all_zips)
print()
print("Demographics fetched for " + str(len(demo_df)) + " zip codes")
demo_df.head()

In [ ]:
# ─────────────────────────────────────────────
# CELL 5: Merge Demographics with Store Data
# ─────────────────────────────────────────────
# Left join so we keep all stores even if Census data is missing for some zips.

merged_df = all_stores.merge(demo_df, on="zip", how="left")

print("Merged dataset shape: " + str(merged_df.shape))
print("Columns: " + str(list(merged_df.columns)))
print()

# Check merge coverage
missing_demo = merged_df["median_income"].isna().sum()
print("Stores missing Census data: " + str(missing_demo) + " (" + str(round(missing_demo/len(merged_df)*100, 1)) + "%)")
merged_df.head()

In [ ]:
# ─────────────────────────────────────────────
# CELL 6: Demographic Comparison by Chain
# ─────────────────────────────────────────────
# Compute average demographics at store zip codes for each chain.
# This reveals the demographic profile each chain targets.

summary = merged_df.groupby("chain").agg(
    store_count          = ("zip", "count"),
    avg_median_income    = ("median_income",    "mean"),
    avg_population       = ("total_population", "mean"),
    avg_median_age       = ("median_age",       "mean"),
    avg_college_plus_pct = ("college_plus_pct", "mean"),
).round(1).reset_index()

summary["avg_median_income"] = summary["avg_median_income"].apply(lambda x: "$" + "{:,.0f}".format(x) if pd.notna(x) else "N/A")

print("=" * 70)
print("DEMOGRAPHIC PROFILE OF STORE ZIP CODES BY CHAIN")
print("=" * 70)
print(summary.to_string(index=False))
print()
print("Interpretation:")
print(" - Higher median income → wealthier neighborhoods")
print(" - Higher college_plus_pct → more educated neighborhoods")
print(" - Sprouts targets health-conscious suburban shoppers")
print(" - Trader Joe's targets 'overeducated and underpaid' urban young adults")
print(" - Whole Foods targets affluent urban/suburban shoppers")

In [ ]:
# ─────────────────────────────────────────────
# CELL 7: State-Level Store Counts
# ─────────────────────────────────────────────
# Compare geographic footprints by state — useful for Tableau map visualizations.

# Normalize state column name
for df_name, df_obj in [("Sprouts", sprouts_df), ("Trader Joes", tj_df), ("Whole Foods", wf_df)]:
    state_col = None
    for c in df_obj.columns:
        if c.lower() in ["state", "st", "state_abbr"]:
            state_col = c
            break
    if state_col and state_col != "state":
        df_obj["state"] = df_obj[state_col]

# State breakdown per chain
for chain_name, df_obj in [("Sprouts", sprouts_df), ("Trader Joes", tj_df), ("Whole Foods", wf_df)]:
    if "state" in df_obj.columns:
        state_counts = df_obj["state"].value_counts()
        print(chain_name + " — stores by state (top 10):")
        print(state_counts.head(10).to_string())
        print("Total states: " + str(df_obj["state"].nunique()))
        print()
    else:
        print(chain_name + ": no 'state' column found")

In [ ]:
# ─────────────────────────────────────────────
# CELL 8: Zip Code Overlap Analysis
# ─────────────────────────────────────────────
# Find zip codes where 2 or more chains compete head-to-head.
# These represent direct competition zones.

sprouts_zips = set(sprouts_df["zip"].dropna())
tj_zips      = set(tj_df["zip"].dropna())
wf_zips      = set(wf_df["zip"].dropna())

# Pairwise overlaps
sp_vs_tj = sprouts_zips & tj_zips
sp_vs_wf = sprouts_zips & wf_zips
tj_vs_wf = tj_zips & wf_zips
all_three = sprouts_zips & tj_zips & wf_zips

print("ZIP CODE COMPETITION OVERLAP")
print("-" * 40)
print("Sprouts vs Trader Joe's:  " + str(len(sp_vs_tj)) + " shared zip codes")
print("Sprouts vs Whole Foods:   " + str(len(sp_vs_wf)) + " shared zip codes")
print("Trader Joe's vs Whole Foods: " + str(len(tj_vs_wf)) + " shared zip codes")
print("All three chains:         " + str(len(all_three)) + " shared zip codes")
print()

# Pull demographics of all-three overlap zones
if all_three:
    overlap_demo = demo_df[demo_df["zip"].isin(all_three)]
    print("Demographics in zones where ALL THREE compete:")
    if len(overlap_demo) > 0:
        print("  Avg median income:     $" + "{:,.0f}".format(overlap_demo["median_income"].mean()))
        print("  Avg college+ pct:      " + str(round(overlap_demo["college_plus_pct"].mean(), 1)) + "%")
        print("  Avg median age:        " + str(round(overlap_demo["median_age"].mean(), 1)))
else:
    print("No zip codes where all three chains co-locate.")

In [ ]:
# ─────────────────────────────────────────────
# CELL 9: Export Final Dataset to CSV
# ─────────────────────────────────────────────
# Export the merged store + demographics data for Tableau.

output_filename = "store_demographics.csv"
merged_df.to_csv(output_filename, index=False)

print("Exported " + str(len(merged_df)) + " rows to " + output_filename)
print("Columns: " + str(list(merged_df.columns)))

# Download from Colab
files.download(output_filename)

# Also export the summary table
summary_filename = "chain_demographic_summary.csv"
summary.to_csv(summary_filename, index=False)
files.download(summary_filename)
print("Also exported summary to " + summary_filename)

## Cell 10: Sprouts Expansion Targets
Identifies zip codes where Trader Joe's or Whole Foods already operate (proving demand) but Sprouts is absent. Scores each zip by how closely its demographics match existing Sprouts locations.

In [ ]:
# ─────────────────────────────────────────────
# CELL 10: Sprouts Expansion Targets
# ─────────────────────────────────────────────
# Find zip codes where TJ or WF exist but Sprouts does NOT.
# Score by demographic similarity to existing Sprouts locations.

sprouts_zips = set(sprouts_df["zip"].dropna())
tj_zips = set(tj_df["zip"].dropna())
wf_zips = set(wf_df["zip"].dropna())

# Zips with a competitor but no Sprouts
target_zips = (tj_zips | wf_zips) - sprouts_zips

# Get demographics for those zips
targets = demo_df[demo_df["zip"].isin(target_zips)].copy()

# Tag which competitor(s) are already there
targets["has_trader_joes"] = targets["zip"].isin(tj_zips)
targets["has_whole_foods"] = targets["zip"].isin(wf_zips)
targets["competitor_count"] = targets["has_trader_joes"].astype(int) + targets["has_whole_foods"].astype(int)

# Score each zip: how close to Sprouts' ideal demographic profile?
# Use the Sprouts chain averages as the "ideal"
sprouts_avg_income = merged_df[merged_df["chain"] == "Sprouts"]["median_income"].mean()
sprouts_avg_college = merged_df[merged_df["chain"] == "Sprouts"]["college_plus_pct"].mean()
sprouts_avg_age = merged_df[merged_df["chain"] == "Sprouts"]["median_age"].mean()

print("Sprouts demographic profile (target baseline):")
print("  Avg median income:     $" + "{:,.0f}".format(sprouts_avg_income))
print("  Avg college+ pct:      " + str(round(sprouts_avg_college, 1)) + "%")
print("  Avg median age:        " + str(round(sprouts_avg_age, 1)))
print()

# Normalized distance score (lower distance = better fit)
targets["income_fit"] = ((targets["median_income"] - sprouts_avg_income) / sprouts_avg_income).abs()
targets["edu_fit"] = ((targets["college_plus_pct"] - sprouts_avg_college) / sprouts_avg_college).abs()
targets["age_fit"] = ((targets["median_age"] - sprouts_avg_age) / sprouts_avg_age).abs()
targets["expansion_score"] = (100 - (targets["income_fit"] + targets["edu_fit"] + targets["age_fit"]) * 33).round(1)
targets["expansion_score"] = targets["expansion_score"].clip(lower=0)

targets = targets.sort_values("expansion_score", ascending=False)

# Export
targets.to_csv("sprouts_expansion_targets.csv", index=False)
files.download("sprouts_expansion_targets.csv")

print("Exported " + str(len(targets)) + " expansion target zip codes.")
print()
print("Top 20 expansion targets:")
print(targets[["zip", "median_income", "college_plus_pct", "median_age", "competitor_count", "expansion_score"]].head(20).to_string(index=False))